# 06 — Logistic Regression + L1 Feature Selection + Grid Search

This notebook uses **Logistic Regression only**. Existing feature tables are reused; no image feature extraction is performed here.

Workflow:
1. Use the already-created **combined forensic feature table** (normal + wavelet + extra descriptors).
2. Fit L1 feature selectors on TRAIN only for a small grid of L1 `C` values.
3. For each selected feature set, run 3-fold `GridSearchCV` for Logistic Regression `C` on TRAIN only.
4. Evaluate the best CV model for each L1 setting on VALIDATION.
5. Select the final configuration by validation ROC-AUC (then PR-AUC).
6. Official TEST is not loaded.

No Random Forest, ExtraTrees, or boosting models are fitted. This keeps the final model compact and suitable for Git.

In [1]:
from pathlib import Path
import sys, gc, json, time
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
 precision_score, recall_score, f1_score, balanced_accuracy_score, confusion_matrix)

ROOT=Path.cwd()
while ROOT!=ROOT.parent and not (ROOT/"data").exists(): ROOT=ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
print("PROJECT_ROOT:",ROOT)
TRAIN_CSV=ROOT/"data/processed/train_combined_features.csv"
VAL_CSV=ROOT/"data/processed/val_combined_features.csv"
assert TRAIN_CSV.exists(), f"Missing: {TRAIN_CSV}"
assert VAL_CSV.exists(), f"Missing: {VAL_CSV}"
from src.features import ALL_COLS
cols=list(ALL_COLS)
tr=pd.read_csv(TRAIN_CSV,usecols=cols+["label"])
va=pd.read_csv(VAL_CSV,usecols=cols+["label"])
ytr=tr.label.to_numpy(dtype=np.int8,copy=False); yv=va.label.to_numpy(dtype=np.int8,copy=False)
Xtr=tr[cols].to_numpy(dtype=np.float32,copy=True)
Xv=va[cols].to_numpy(dtype=np.float32,copy=True)
print("Train rows:",len(tr),"Validation rows:",len(va),"Combined features:",len(cols))

def evaluate(y,p,threshold=.5):
    pred=np.asarray(p)>=threshold
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {"roc_auc":roc_auc_score(y,p),"pr_auc":average_precision_score(y,p),
            "accuracy":accuracy_score(y,pred),"precision":precision_score(y,pred,zero_division=0),
            "recall":recall_score(y,pred,zero_division=0),"f1":f1_score(y,pred,zero_division=0),
            "balanced_accuracy":balanced_accuracy_score(y,pred),"tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp)}

L1_C_VALUES=[0.001,0.003,0.01,0.03,0.1,0.3,1.0]
MODEL_C_VALUES=[0.01,0.03,0.1,0.3,1.0,3.0,10.0]
cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=42)
rows=[]; selections=[]

for l1_C in L1_C_VALUES:
    print("\n"+"="*70); print("L1 C:",l1_C); print("="*70)
    selector=Pipeline([
        ("imputer",SimpleImputer(strategy="median")),
        ("scaler",StandardScaler()),
        ("l1",LogisticRegression(penalty="l1",solver="liblinear",C=l1_C,max_iter=3000,class_weight="balanced",random_state=42))
    ])
    t0=time.perf_counter(); selector.fit(Xtr,ytr)
    coef=selector.named_steps["l1"].coef_[0]
    mask=coef!=0
    selected=[c for c,m in zip(cols,mask) if m]
    print("Selected features:",len(selected),"L1 fit seconds:",round(time.perf_counter()-t0,1))
    if not selected:
        print("No features selected; skipping."); del selector,coef,mask; gc.collect(); continue
    idx=[cols.index(c) for c in selected]
    Xs=np.ascontiguousarray(Xtr[:,idx],dtype=np.float32)
    Xvs=np.ascontiguousarray(Xv[:,idx],dtype=np.float32)
    model_pipe=Pipeline([
        ("imputer",SimpleImputer(strategy="median")),
        ("scaler",StandardScaler()),
        ("model",LogisticRegression(solver="liblinear",max_iter=3000,class_weight="balanced",random_state=42))
    ])
    search=GridSearchCV(model_pipe,{"model__C":MODEL_C_VALUES},scoring="roc_auc",cv=cv,n_jobs=1,refit=True,return_train_score=False)
    t1=time.perf_counter(); search.fit(Xs,ytr)
    p=search.best_estimator_.predict_proba(Xvs)[:,1]
    e=evaluate(yv,p)
    print("Best model C:",search.best_params_["model__C"],"CV ROC-AUC:",round(search.best_score_,6))
    print("Validation:",json.dumps(e))
    rows.append({"feature_set":"all_forensic","l1_C":l1_C,"selected_features":len(selected),
                 "model":"logistic_regression","model_C":float(search.best_params_["model__C"]),
                 "cv_roc_auc":float(search.best_score_),"grid_fit_seconds":float(time.perf_counter()-t1),**e})
    selections.append({"feature_set":"all_forensic","l1_C":l1_C,"selected_features":len(selected),"features":selected})
    del selector,coef,mask,Xs,Xvs,search,model_pipe,p,e
    gc.collect()

assert rows,"No Logistic Regression configuration completed."
res=pd.DataFrame(rows).sort_values(["roc_auc","pr_auc"],ascending=False).reset_index(drop=True)
metrics_dir=ROOT/"metrics"; metrics_dir.mkdir(exist_ok=True)
res.to_csv(metrics_dir/"l1_model_comparison.csv",index=False)
(metrics_dir/"l1_selected_features.json").write_text(json.dumps(selections,indent=2),encoding="utf-8")
display(res)
best=res.iloc[0].to_dict()
print("\nBEST VALIDATION CONFIGURATION")
print(json.dumps(best,indent=2,default=str))
print("Saved:",metrics_dir/"l1_model_comparison.csv")
print("Saved:",metrics_dir/"l1_selected_features.json")
print("\nTEST STATUS: official test was not loaded.")


PROJECT_ROOT: d:\deepfake_noise_wavelet_ml
Train rows: 230361 Validation rows: 48289 Combined features: 105

L1 C: 0.001
Selected features: 28 L1 fit seconds: 25.8
Best model C: 10.0 CV ROC-AUC: 0.658843
Validation: {"roc_auc": 0.6914737797420691, "pr_auc": 0.7597053897797017, "accuracy": 0.6457992503468699, "precision": 0.7033010405453893, "recall": 0.6893022928681952, "f1": 0.6962313074983127, "balanced_accuracy": 0.6363954671916657, "tn": 11584, "fp": 8269, "fn": 8835, "tp": 19601}

L1 C: 0.003
Selected features: 52 L1 fit seconds: 87.3
Best model C: 10.0 CV ROC-AUC: 0.692796
Validation: {"roc_auc": 0.7062520600403682, "pr_auc": 0.7721243114519396, "accuracy": 0.6594462506989169, "precision": 0.7157042633567189, "recall": 0.6995709663806442, "f1": 0.7075456598673329, "balanced_accuracy": 0.6507727395243774, "tn": 11951, "fp": 7902, "fn": 8543, "tp": 19893}

L1 C: 0.01
Selected features: 72 L1 fit seconds: 740.7
Best model C: 10.0 CV ROC-AUC: 0.69979
Validation: {"roc_auc": 0.7106253

,feature_set,l1_C,selected_features,model,model_C,cv_roc_auc,grid_fit_seconds,roc_auc,pr_auc,accuracy,precision,recall,f1,balanced_accuracy,tn,fp,fn,tp
0,all_forensic,0.100,100,logistic_regression,10.0,0.704602,3346.839014,0.714856,0.782357,0.666529,0.720413,0.708785,0.714552,0.657394,12031,7822,8281,20155
1,all_forensic,0.300,101,logistic_regression,10.0,0.704814,1051.811565,0.713684,0.781271,0.665100,0.719235,0.707448,0.713293,0.655945,12000,7853,8319,20117
2,all_forensic,1.000,102,logistic_regression,10.0,0.704831,939.631214,0.713642,0.781252,0.665120,0.719339,0.707272,0.713255,0.656009,12006,7847,8324,20112
3,all_forensic,0.030,90,logistic_regression,10.0,0.703696,479.001320,0.712575,0.780984,0.664478,0.720703,0.702455,0.711462,0.656269,12112,7741,8461,19975
4,all_forensic,0.010,72,logistic_regression,10.0,0.699790,821.370697,0.710625,0.779897,0.663671,0.718697,0.704670,0.711615,0.654808,12010,7843,8398,20038
5,all_forensic,0.003,52,logistic_regression,10.0,0.692796,170.456487,0.706252,0.772124,0.659446,0.715704,0.699571,0.707546,0.650773,11951,7902,8543,19893
6,all_forensic,0.001,28,logistic_regression,10.0,0.658843,76.166137,0.691474,0.759705,0.645799,0.703301,0.689302,0.696231,0.636395,11584,8269,8835,19601



BEST VALIDATION CONFIGURATION
{
  "feature_set": "all_forensic",
  "l1_C": 0.1,
  "selected_features": 100,
  "model": "logistic_regression",
  "model_C": 10.0,
  "cv_roc_auc": 0.7046022932313093,
  "grid_fit_seconds": 3346.839014399913,
  "roc_auc": 0.7148558273403764,
  "pr_auc": 0.7823566120240091,
  "accuracy": 0.6665286089999793,
  "precision": 0.7204131965543125,
  "recall": 0.7087846391897594,
  "f1": 0.7145516104444011,
  "balanced_accuracy": 0.6573943847739459,
  "tn": 12031,
  "fp": 7822,
  "fn": 8281,
  "tp": 20155
}
Saved: d:\deepfake_noise_wavelet_ml\metrics\l1_model_comparison.csv
Saved: d:\deepfake_noise_wavelet_ml\metrics\l1_selected_features.json

TEST STATUS: official test was not loaded.
